In [ ]:
import json
import re
import difflib
from pathlib import Path
from typing import List, Optional

class Contact:
    """کلاس مدل برای نگهداری اطلاعات یک مخاطب"""
    def __init__(self, contact_id: int, name: str, phone: str, email: str, address: str):
        self.contact_id = contact_id
        self.name = name
        self.phone = phone
        self.email = email
        self.address = address

    def to_dict(self) -> dict:
        return {
            "id": self.contact_id,
            "name": self.name,
            "phone": self.phone,
            "email": self.email,
            "address": self.address
        }

    @classmethod
    def from_dict(cls, data: dict) -> 'Contact':
        return cls(data["id"], data["name"], data["phone"], data["email"], data["address"])

    def __str__(self) -> str:
        return f"ID: {self.contact_id:03d} |  {self.name:<15} |  {self.phone:<15} |  {self.email:<20} |  {self.address}"

#کلاس مدیریت و جست و جو هوشمند
class ContactManager:
    def __init__(self, filename: str = "contacts_data.json"):
        self.filename = filename
        self.contacts: List[Contact] = []
        self.next_id = 1
        self.load_data()

    def load_data(self):
      
        file_path = Path(self.filename)
        
        #بررسی وجود فایل 
        if file_path.is_file():
            with open(self.filename, 'r', encoding='utf-8') as file:
                data = json.load(file)
                self.contacts = [Contact.from_dict(item) for item in data]
                if self.contacts:
                    self.next_id = max(c.contact_id for c in self.contacts) + 1

    def save_data(self):
        #ذخیره اطلاعات در فایل
        with open(self.filename, 'w', encoding='utf-8') as file:
            data = [contact.to_dict() for contact in self.contacts]
            json.dump(data, file, ensure_ascii=False, indent=4)

    def add_contact(self, name: str, phone: str, email: str, address: str):
        new_contact = Contact(self.next_id, name, phone, email, address)
        self.contacts.append(new_contact)
        self.next_id += 1
        self.save_data()
        print(" .مخاطب با موفقیت اضافه شد")

    def get_all_contacts(self) -> List[Contact]:
        return self.contacts

    def search_contacts(self, query: str) -> List[Contact]:
        #difflib 
        results = []
        query_lower = query.lower()

        # 1. بررسی تطابق دقیق یا بخشی از نام/تلفن 
        for c in self.contacts:
            if query_lower in c.name.lower() or query_lower in c.phone:
                if c not in results:
                    results.append(c)

        # 2.  (Fuzzy Search) 
        names = [c.name for c in self.contacts]
        close_matches = difflib.get_close_matches(query, names, n=3, cutoff=0.4)
        
        for match in close_matches:
            for c in self.contacts:
                if c.name == match and c not in results:
                    results.append(c)

        return results

    def get_contact_by_id(self, contact_id: int) -> Optional[Contact]:
        for contact in self.contacts:
            if contact.contact_id == contact_id:
                return contact
        return None

    def update_contact(self, contact_id: int, name: str, phone: str, email: str, address: str) -> bool:
        contact = self.get_contact_by_id(contact_id)
        if contact:
            contact.name = name if name else contact.name
            contact.phone = phone if phone else contact.phone
            contact.email = email if email else contact.email
            contact.address = address if address else contact.address
            self.save_data()
            return True
        return False

    def delete_contact(self, contact_id: int) -> bool:
        contact = self.get_contact_by_id(contact_id)
        if contact:
            self.contacts.remove(contact)
            self.save_data()
            return True
        return False


# 3. اعتبارسنجی با Regex

def validate_email(email: str) -> bool:
    """Regexاعتبارسنجی ایمیل با استفاده از """
    if not email: return True 
    pattern = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$"
    return bool(re.match(pattern, email))

def validate_phone(phone: str) -> bool:
    """Regexاعتبارسنجی شماره موبایل با استفاده از """
    if not phone: return True
    pattern = r"^(09\d{9}|\+?\d{10,14})$"
    return bool(re.match(pattern, phone))

def get_input(prompt: str, required: bool = False, validator=None, error_msg: str = ".ورودی نامعتبر است") -> str:
    while True:
        value = input(prompt).strip()
        if required and not value:
            print(" .این فیلد نمی‌تواند خالی باشد")
            continue
        if validator and not validator(value):
            print(f" {error_msg}")
            continue
        return value

def print_contacts(contacts: List[Contact]):
    if not contacts:
        print("\n .هیچ مخاطبی یافت نشد")
        return
    print("-" * 85)
    for c in contacts:
        print(c)
    print("-" * 85)

def main():
    manager = ContactManager()
    
    while True:
        print("\n" + "="*40)
        print("  سیستم مدیریت مخاطبین هوشمند  ")
        print("="*40)
        print("1.  افزودن مخاطب جدید")
        print("2.  نمایش همه مخاطبین")
        print("3.  جستجوی هوشمند (Fuzzy Search)")
        print("4.  ویرایش مخاطب")
        print("5.  حذف مخاطب")
        print("6.  خروج")
        print("="*40)

        choice = input("انتخاب شما (1-6): ").strip()

        if choice == '1':
            print("\n افزودن مخاطب جدید ")
            name = get_input("نام و نام خانوادگی (اجباری): ", required=True)
            phone = get_input("شماره موبایل ): ", validator=validate_phone, error_msg=".شماره موبایل نامعتبر است")
            email = get_input("ایمیل (اختیاری): ", validator=validate_email, error_msg=".فرمت ایمیل نامعتبر است")
            address = get_input("آدرس (اختیاری): ")
            manager.add_contact(name, phone, email, address)

        elif choice == '2':
            print("\n لیست تمام مخاطبین ")
            print_contacts(manager.get_all_contacts())

        elif choice == '3':
            print("\n جستجوی هوشمند ")
            print("راهنمایی: می‌توانید نام را ناقص یا با غلط املایی وارد کنید")
            query = input("عبارت جستجو: ")
            results = manager.search_contacts(query)
            print_contacts(results)

        elif choice == '4':
            print("\n ویرایش مخاطب ")
            cid_str = input(" مخاطب برای ویرایش را وارد کنیدID: ").strip()
            
        
            if not cid_str.isdigit():
                print("\.شناسه باید یک عدد باشد")
                continue
                
            cid = int(cid_str)
            if not manager.get_contact_by_id(cid):
                print(" .مخاطبی با این شناسه یافت نشد")
                continue
            
            print(" .بزنید Enter راهنمایی: اگر نمی‌خواهید فیلدی تغییر کند، آن را خالی بگذارید و سپس  ")
            name = input("نام جدید: ")
            phone = get_input("شماره موبایل جدید: ", validator=validate_phone, error_msg=".شماره موبایل نامعتبر است")
            email = get_input("ایمیل جدید: ", validator=validate_email, error_msg=".فرمت ایمیل نامعتبر است")
            address = input("آدرس جدید: ")
            
            if manager.update_contact(cid, name, phone, email, address):
                print(" .اطلاعات مخاطب با موفقیت بروزرسانی شد")

        elif choice == '5':
            print("\n حذف مخاطب ")
            cid_str = input("مخاطب برای حذف را وارد کنید Id: ").strip()
            
            if not cid_str.isdigit():
                print(" .شناسه باید یک عدد باشد")
                continue
                
            cid = int(cid_str)
            contact = manager.get_contact_by_id(cid)
            if contact:
                confirm = input(f" اطمینان داری؟{contact.self}آیا از حذف : ").lower()
                if confirm == 'y':
                    manager.delete_contact(cid)
                    print(" .مخاطب با موفقیت حذف شد")
                else:
                    print(".عملیات حذف لغو شد")
            else:
                print(" مخاطبی با این شناسه یافت نشد")

        elif choice == '6':
            print("\n  .اطلاعات ذخیره شد")
            break
        else:
            print("  .لطفاً عددی بین 1 تا 6 وارد کنید")
